# 3. Microsoft Sentinel — Implementation

Sentinel is Microsoft's cloud-native **SIEM + SOAR**:

* **SIEM** — Security Information & Event Management: ingest, normalize and query logs.
* **SOAR** — Security Orchestration, Automation & Response: run playbooks to contain incidents.

Under the hood, Sentinel is a **solution layered on a Log Analytics workspace** + a library of analytics rules + Logic Apps for automation.

### What you'll build

1. A **mini KQL interpreter** in ~30 lines of Python so KQL stops feeling magical.
2. **Analytics rules** that evaluate against a simulated event stream and create incidents.
3. **Entity mapping** + **MITRE ATT&CK tactic** grouping on those incidents.
4. A **Logic App-style playbook** that fires per incident.
5. A **bad → best** Sentinel rollout.

## Before you run this notebook

1. From the lab folder run `uv sync`.
2. In VS Code pick the `.venv` kernel (top-right kernel picker).
3. Reload the window if the kernel doesn't appear.

Everything is simulated — no Azure subscription required.

## 1. KQL, demystified

Every KQL query is a pipeline:

```
TableName
| where <filter>
| summarize <agg> by <group_cols>, bin(TimeGenerated, <window>)
| where <filter_on_agg>
| project <columns>
```

It reads top-to-bottom. Each `|` hands the rows from the previous step to the next step. That's it. You can build 80% of Sentinel rules with just `where`, `summarize`, `bin`, `count()`, and `project`.

Let's build a *toy* interpreter for exactly that subset to prove it.

In [1]:
from datetime import datetime, timedelta
from collections import defaultdict

# A handful of simulated SigninLogs rows
signin_logs = [
    {"Time": datetime(2026, 4, 20, 9,  1), "User": "alice",  "IP": "10.0.0.10",       "ResultType": 0},
    {"Time": datetime(2026, 4, 20, 9,  2), "User": "bob",    "IP": "203.0.113.7",     "ResultType": 50126},
    {"Time": datetime(2026, 4, 20, 9,  3), "User": "bob",    "IP": "203.0.113.7",     "ResultType": 50126},
    {"Time": datetime(2026, 4, 20, 9,  4), "User": "bob",    "IP": "203.0.113.7",     "ResultType": 50126},
    {"Time": datetime(2026, 4, 20, 9,  5), "User": "bob",    "IP": "203.0.113.7",     "ResultType": 50126},
    {"Time": datetime(2026, 4, 20, 9,  6), "User": "bob",    "IP": "203.0.113.7",     "ResultType": 50126},
    {"Time": datetime(2026, 4, 20, 9,  7), "User": "bob",    "IP": "203.0.113.7",     "ResultType": 50126},
    {"Time": datetime(2026, 4, 20, 9,  8), "User": "bob",    "IP": "203.0.113.7",     "ResultType": 50126},
    {"Time": datetime(2026, 4, 20, 9, 14), "User": "carol",  "IP": "198.51.100.9",    "ResultType": 50126},
    {"Time": datetime(2026, 4, 20, 9, 20), "User": "alice",  "IP": "10.0.0.10",       "ResultType": 0},
]


def where(rows, predicate):
    return [r for r in rows if predicate(r)]

def summarize(rows, group_by, aggregators):
    # aggregators: {name: (op, col)} with op in {"count","sum","max"}
    buckets = defaultdict(list)
    for r in rows:
        key = tuple(r[c] if not callable(c) else c(r) for c in group_by)
        buckets[key].append(r)
    out = []
    for key, items in buckets.items():
        row = {name: v if not callable(v) else v(key) for name, v in zip(group_by_names(group_by), key)}
        for name, (op, col) in aggregators.items():
            if op == "count": row[name] = len(items)
            elif op == "sum": row[name] = sum(i[col] for i in items)
            elif op == "max": row[name] = max(i[col] for i in items)
        out.append(row)
    return out

def group_by_names(group_by):
    return [g.__name__ if callable(g) else g for g in group_by]

def bin_time(col, minutes):
    def f(r):
        t = r[col]
        whole = t.replace(second=0, microsecond=0, minute=(t.minute // minutes) * minutes)
        return whole
    f.__name__ = f"bin_{col}_{minutes}m"
    return f

# KQL:
# SigninLogs
# | where ResultType != 0 and IP !startswith "10."
# | summarize Fails = count() by User, IP, bin(Time, 10m)
# | where Fails >= 5
step1 = where(signin_logs, lambda r: r["ResultType"] != 0 and not r["IP"].startswith("10."))
step2 = summarize(step1, group_by=["User", "IP", bin_time("Time", 10)], aggregators={"Fails": ("count", None)})
step3 = where(step2, lambda r: r["Fails"] >= 5)

print("=== Suspicious sign-ins detected ===")
for r in step3:
    print(f"  user={r['User']:6s} ip={r['IP']:14s} window={r['bin_Time_10m']:%H:%M} fails={r['Fails']}")


=== Suspicious sign-ins detected ===
  user=bob    ip=203.0.113.7    window=09:00 fails=7


### Reading that back

* `where` threw out successful sign-ins and anything from the corporate range.
* `summarize ... by User, IP, bin(Time, 10m)` bucketed the rest into 10-minute windows per user/IP.
* `where Fails >= 5` kept only the windows with 5+ failures — Bob's brute force.

That's the exact pattern every Sentinel detection uses. Rebuild any real KQL rule by asking: *which filter, which summarize, which threshold?*

## 2. Analytics rules — detect, then create incidents

An analytics rule is a KQL query + a schedule + a threshold. When it returns rows, Sentinel creates an **alert**, and several alerts can be grouped into an **incident**.

In [2]:
ANALYTICS_RULES = [
    {
        "name": "Brute-force sign-in",
        "severity": "High",
        "tactics": ["CredentialAccess", "InitialAccess"],
        "description": ">= 5 failed Entra sign-ins from the same external IP in 10 min.",
        "frequency_min": 5,
        "lookback_min": 10,
        "detect": lambda rows: where(
            summarize(
                where(rows, lambda r: r["ResultType"] != 0 and not r["IP"].startswith("10.")),
                group_by=["User", "IP", bin_time("Time", 10)],
                aggregators={"Fails": ("count", None)},
            ),
            lambda r: r["Fails"] >= 5,
        ),
        "entities": lambda hit: {"account": hit["User"], "ip": hit["IP"]},
    },
    {
        "name": "Impossible travel",
        "severity": "Medium",
        "tactics": ["InitialAccess"],
        "description": "Same user, two successful sign-ins from countries > 500 km apart inside an hour.",
        "frequency_min": 30,
        "lookback_min": 60,
        # Stubbed out — we'd need geo data; left here to show how rules are structured.
        "detect": lambda rows: [],
        "entities": lambda hit: {},
    },
]


def run_rule(rule, rows):
    hits = rule["detect"](rows)
    alerts = []
    for h in hits:
        alerts.append({
            "rule": rule["name"],
            "severity": rule["severity"],
            "tactics": rule["tactics"],
            "entities": rule["entities"](h),
            "raw": h,
        })
    return alerts


all_alerts = []
for rule in ANALYTICS_RULES:
    alerts = run_rule(rule, signin_logs)
    print(f"• {rule['name']:25s} → {len(alerts)} alert(s)")
    all_alerts.extend(alerts)


• Brute-force sign-in       → 1 alert(s)
• Impossible travel         → 0 alert(s)


## 3. From alerts to incidents — entity mapping & ATT&CK

Sentinel groups alerts into incidents using:

* The same **entities** (user, IP, host, file, …) across alerts.
* A **grouping policy** (e.g. "group by user over the last 5 hours").
* **MITRE ATT&CK tactics** get stamped on both the alert and the incident.

In [3]:
from collections import defaultdict
import random

random.seed(42)

def build_incidents(alerts, group_by_entity="account", window_min=30):
    buckets = defaultdict(list)
    for a in alerts:
        key = a["entities"].get(group_by_entity, "unknown")
        buckets[key].append(a)
    incidents = []
    for key, group in buckets.items():
        incidents.append({
            "id": f"INC-{random.randint(1000,9999)}",
            "title": f"{group[0]['rule']} involving {key}",
            "status": "New",
            "severity": max(group, key=lambda a: {"High": 3, "Medium": 2, "Low": 1}[a["severity"]])["severity"],
            "tactics": sorted({t for a in group for t in a["tactics"]}),
            "entities": {k: v for a in group for k, v in a["entities"].items()},
            "alert_count": len(group),
        })
    return incidents


incidents = build_incidents(all_alerts)
print("=== Incidents ===")
for inc in incidents:
    sev_icon = {"High": "🔴", "Medium": "🟡", "Low": "🟢"}[inc["severity"]]
    print(f"  {sev_icon} {inc['id']} [{inc['severity']}] {inc['title']}")
    print(f"       tactics : {', '.join(inc['tactics'])}")
    print(f"       entities: {inc['entities']}")
    print(f"       alerts  : {inc['alert_count']}\n")


=== Incidents ===
  🔴 INC-2824 [High] Brute-force sign-in involving bob
       tactics : CredentialAccess, InitialAccess
       entities: {'account': 'bob', 'ip': '203.0.113.7'}
       alerts  : 1



## 4. Playbooks — Logic Apps for automated response

A **playbook** is a Logic App triggered by a Sentinel **automation rule**:

```
incident (created|updated)  ─▶  automation rule (filter)  ─▶  playbook (Logic App)
```

Playbooks typically do some combination of: block IP at the firewall, disable the user, tag the incident, post to Teams, open a ServiceNow ticket, call an EDR containment action.

Below is a (Python-simulated) playbook library keyed by the rule name.

In [4]:
PLAYBOOKS = {
    "Brute-force sign-in": [
        ("Block source IP in Azure Firewall",        lambda inc: f"az network firewall ip-rule add --ip {inc['entities']['ip']}"),
        ("Disable user in Entra ID",                 lambda inc: f"az ad user update --id {inc['entities']['account']} --account-enabled false"),
        ("Revoke user sessions",                     lambda inc: f"Revoke-MgUserSignInSession -UserId {inc['entities']['account']}"),
        ("Post to SOC Teams channel",                lambda inc: f"POST https://teams.example/socs {{'incident': '{inc['id']}'}}"),
        ("Open ServiceNow incident P2",              lambda inc: f"POST https://now.example/api/incident {{'short_description': '{inc['title']}'}}"),
    ],
    "Impossible travel": [
        ("Force MFA re-prompt",                      lambda inc: f"Require MFA for {inc['entities'].get('account', '?')}"),
        ("Notify user's manager",                    lambda inc: f"Email manager of {inc['entities'].get('account', '?')}"),
    ],
}


def run_playbook(incident):
    actions = PLAYBOOKS.get(incident["title"].split(" involving")[0], [])
    if not actions:
        return [("Manual investigation required", "")]
    return [(name, fn(incident)) for name, fn in actions]


for inc in incidents:
    print(f"▶ Playbook run for {inc['id']} — {inc['title']}")
    for i, (step, cmd) in enumerate(run_playbook(inc), 1):
        print(f"    {i}. ✅ {step}")
        if cmd:
            print(f"       $ {cmd}")
    print()


▶ Playbook run for INC-2824 — Brute-force sign-in involving bob
    1. ✅ Block source IP in Azure Firewall
       $ az network firewall ip-rule add --ip 203.0.113.7
    2. ✅ Disable user in Entra ID
       $ az ad user update --id bob --account-enabled false
    3. ✅ Revoke user sessions
       $ Revoke-MgUserSignInSession -UserId bob
    4. ✅ Post to SOC Teams channel
       $ POST https://teams.example/socs {'incident': 'INC-2824'}
    5. ✅ Open ServiceNow incident P2
       $ POST https://now.example/api/incident {'short_description': 'Brute-force sign-in involving bob'}



## 5. Data connectors & DCRs — where the logs come from

| Connector                     | Data                                        | Setup                                |
|-------------------------------|---------------------------------------------|--------------------------------------|
| **Microsoft Entra ID**        | Sign-in & audit logs                        | 1-click in Sentinel                  |
| **Microsoft 365 / Defender XDR** | Exchange / SharePoint / Teams / XDR alerts  | 1-click                              |
| **Azure Activity**            | ARM operations                              | Diagnostic setting → LA              |
| **Azure resources** (KV, SQL, Firewall, NSG flow) | per-resource logs         | Diagnostic setting → LA              |
| **AWS / GCP**                 | CloudTrail / GCP audit logs                 | Native connectors (S3-based for AWS) |
| **CEF / Syslog**              | Third-party firewalls, Linux                | Forwarder VM running AMA + DCR       |
| **Custom logs** (any JSON)    | App logs                                    | **Data Collection Rule (DCR)** + API |

```bash
# Enable the Azure Activity connector by sending diagnostics to the workspace
az monitor diagnostic-settings create -n sentinel-activity \
  --resource /subscriptions/<sub-id> \
  --workspace la-sentinel \
  --logs '[{"category":"Administrative","enabled":true},
          {"category":"Security","enabled":true},
          {"category":"Alert","enabled":true}]'

# Data Collection Rule: grab Windows security events → send to Sentinel workspace
az monitor data-collection rule create -g rg-security -n dcr-sec-events \
  --location eastus \
  --data-sources '{"windowsEventLogs":[{"name":"sec","streams":["Microsoft-SecurityEvent"],
       "xPathQueries":["Security!*[System[(EventID=4624 or EventID=4625 or EventID=4688)]]"]}]}' \
  --data-flows '[{"streams":["Microsoft-SecurityEvent"],"destinations":["la-sentinel"]}]' \
  --destinations '{"logAnalytics":[{"name":"la-sentinel","workspaceResourceId":"/subscriptions/.../workspaces/la-sentinel"}]}'
```

**Exam tip**: **AMA (Azure Monitor Agent) + DCR** is the supported path. The old *Log Analytics agent (MMA)* is deprecated.

## 6. Creating analytics rules via CLI

```bash
az sentinel alert-rule create -g rg-security \
  --workspace-name la-sentinel --rule-name brute-force-ssh \
  --type Scheduled \
  --display-name "Brute force SSH" \
  --description "Detects >10 failed SSH attempts per hour" \
  --severity High \
  --query 'Syslog | where Facility=="auth" and SyslogMessage contains "Failed password"
           | summarize c=count() by HostIP, bin(TimeGenerated, 1h)
           | where c > 10' \
  --query-frequency PT1H --query-period PT1H \
  --trigger-operator GreaterThan --trigger-threshold 0 \
  --tactics CredentialAccess \
  --enabled true
```

### Five rule **types** you should recognize

| Type               | When it fires                                                    |
|--------------------|------------------------------------------------------------------|
| **Scheduled**      | A KQL query on a schedule (the most common)                      |
| **Microsoft Security** | Passes alerts from another Microsoft product (Defender XDR)  |
| **Fusion**         | ML that correlates multi-stage attacks across products           |
| **ML Behavior**    | Built-in anomaly detections (e.g. impossible travel)             |
| **NRT (Near-Real-Time)** | KQL evaluated every ~1 minute; limited to a single event |

## 7. Bad → best: Sentinel rollout

```bash
# ❌ BAD — workspace created, no connectors, no rules. "We have a SIEM!"
az monitor log-analytics workspace create -g rg-security -n la-sentinel --sku PerGB2018

# 🟡 BETTER — Sentinel onboarded + key connectors + Microsoft-provided rule templates.
az sentinel onboarding-state create -g rg-security --workspace-name la-sentinel -n default
# Then: enable Entra ID, Defender XDR, Azure Activity connectors in the portal.

# 🟢 BEST — UEBA on, all Entra & Defender rules enabled, automation rules route to playbooks,
#   watchlists for VIPs, content hub solutions for each key workload (Entra, M365, KV, Firewall),
#   retention tuned (interactive 90d, archive 2y), SOC runbooks linked to each rule.
az sentinel automation-rule create -g rg-security --workspace-name la-sentinel \
  --automation-rule-name auto-contain-high \
  --order 1 \
  --triggering-logic '{"isEnabled": true, "triggersOn":"Incidents", "triggersWhen":"Created",
       "conditions":[{"conditionType":"Property",
         "conditionProperties":{"propertyName":"IncidentSeverity","operator":"Equals","propertyValues":["High"]}}]}' \
  --actions '[{"actionType":"RunPlaybook",
       "actionConfiguration":{"logicAppResourceId":"/subscriptions/.../playbook-contain"}}]'
```

### One-screen AZ-500 Sentinel cheat-sheet

| Thing                      | Key detail                                                                         |
|----------------------------|------------------------------------------------------------------------------------|
| Foundation                 | **Log Analytics workspace** + Sentinel solution.                                   |
| Ingestion                  | Data connectors + AMA/DCRs. Legacy MMA is deprecated.                              |
| Detection                  | Analytics rules: Scheduled, NRT, MS Security, Fusion, ML Behavior.                 |
| KQL backbone               | `where` → `summarize` → `bin` → `where` → `project`.                               |
| Incident grouping          | By entities + grouping policy. Tactics from MITRE ATT&CK.                          |
| Automation                 | Automation rule → Logic App playbook.                                              |
| Investigation tools        | Investigation graph, UEBA, Hunting queries, Notebooks, Workbooks.                  |
| Defender for Cloud ↔ Sentinel | Enable the Microsoft Defender for Cloud connector so alerts flow both ways.     |

## You've completed all AZ-500 labs!

### Next steps

1. Take the [AZ-500 practice assessment](https://learn.microsoft.com/en-us/credentials/certifications/azure-security-engineer/practice/assessment?assessment-type=practice&assessmentId=57&practice-assessment-type=certification).
2. Deploy the Azure CLI examples in a free Azure subscription — the CLI skills transfer directly.
3. Read the [Microsoft Cloud Security Benchmark](https://learn.microsoft.com/en-us/security/benchmark/azure/overview).
4. Skim the [Sentinel content hub](https://learn.microsoft.com/en-us/azure/sentinel/sentinel-solutions) to see how much is provided out of the box.